# Payload (ETC) Functional Test Analysis

Analyzes focus, sharpness, and astigmatism in payload camera test captures across functional test runs (pre/post vibe, TVAC thermal cycling). See `src/payload_analysis/` for the underlying functions — this notebook is just the run log.

In [ ]:
import sys
sys.path.insert(0, '../src')

from payload_analysis import (
    analyze_payload_image,
    analyze_payload_directory,
    compute_focus_scores,
    analyze_astigmatism_by_angle,
    plot_payload_metrics,
    plot_payload_metrics_timeline,
    plot_focus_scores,
    add_temperatures_and_plot,
    DATA_DIR,
)


## Single-image sanity check

Quick look at one capture before running a full batch, to confirm the metrics look reasonable.

In [ ]:
sample_image = DATA_DIR / 'sample_capture.jpg'  # swap in a real capture path
if sample_image.exists():
    print(analyze_payload_image(str(sample_image)))
else:
    print(f"No sample image at {sample_image} — skipping sanity check")


## Pre/post-vibe functional test runs

Each call processes all `angle-*` subdirectories in a test-run folder and writes a metrics CSV to `results/`.

In [ ]:
vibe_test_dirs = [
    DATA_DIR / '260609_pre_vibe_func_test',
    DATA_DIR / '260612_post_vibe_func_test',
    DATA_DIR / '260615_pre_vibe2_func_test',
    DATA_DIR / '260616_post_vibe2_func_test',
]

for test_dir in vibe_test_dirs:
    analyze_payload_directory(test_dir)


In [ ]:
from payload_analysis.config import RESULTS_DIR

vibe_csvs = [RESULTS_DIR / f'{d.name}_all_metrics.csv' for d in vibe_test_dirs]
for csv_path in vibe_csvs:
    plot_payload_metrics(csv_path)


### Timeline across vibe test runs

In [ ]:
timeline_df = plot_payload_metrics_timeline(vibe_csvs)
timeline_df


### Astigmatism: target tilt vs. optical

If astigmatism ratio varies a lot by angle, it's likely target tilt rather than a real optical astigmatism. See `analyze_astigmatism_by_angle` for the variance threshold used.

In [ ]:
analyze_astigmatism_by_angle(vibe_csvs[2])  # 260615_pre_vibe2_func_test


## TRP (thermal cycling) run

Same metric set, run over the full TRP master directory (one CSV per run instead of per angle-subset).

In [ ]:
trp_dir = DATA_DIR / 'New TRP'
trp_df = analyze_payload_directory(trp_dir)


In [ ]:
trp_summary = plot_payload_metrics(RESULTS_DIR / 'New TRP_all_metrics.csv')


## LoG focus score vs. temperature

Independent focus metric (LoG-based) as a cross-check on the Laplacian-variance metric above, plotted against operating temperature per `config.TEMP_MAP`.

In [ ]:
focus_df = compute_focus_scores(trp_dir)
plot_focus_scores(focus_df)


In [ ]:
focus_df_with_temp = add_temperatures_and_plot(focus_df)
focus_df_with_temp[['subdir', 'temperature', 'focus_score']].sort_values('temperature')
